# S6 — Plotly 互動 + 完整 Capstone 案例【解答版】

> **版本**：解答版（含 3 個延伸挑戰）  
> **時間**：2 小時（講授 60 + 實作 50 + 成果發表 10）  
> **資料**：`orders_raw.csv`（從頭走一遍完整 DE/DA 流程）  
> **學完能幹嘛**：獨立完成 raw CSV → 清理 → 分析 → 互動式儀表板的**全流程**，並且可以寫出 **可重用的儀表板產生函式**。

---

## 解答版的三個重點

本 notebook 在 S6 原版的基礎上額外提供：

1. **延伸挑戰 1：時間篩選器** — 使用 `plotly.graph_objects` 的 `updatemenus` 在月度趨勢圖上加入「全年 / Q1 / Q2 / Q3 / Q4」切換按鈕。
2. **延伸挑戰 2：RFM 3D 散佈圖** — 用 `px.scatter_3d` 將顧客依 Recency / Frequency / Monetary 分群，色彩依 `vip_level` 區分。
3. **延伸挑戰 3：可重用 Pipeline** — 把「載入 → 清理 → 合併 → 儀表板 → 存 HTML」整個打包成 `generate_dashboard()` 函式，未來只要換 CSV 路徑就能跑出新報表。

解答版同時會**把 S6 的教學提示補得更完整**，特別是 Plotly `subplots.make_subplots` 的 `specs` 參數、Plotly Express 在複雜排版時的限制、以及 HTML 成品該如何交付 / 部署。

---
## Part A — Plotly Express 速成（30 min）

先複習 6 個最常用的 Plotly Express 函式。這一段與 S6 完全相同，目的只是讓你在動手做 Capstone 前，手感先熱起來。

> 💡 **教學提示**：Plotly Express（簡稱 `px`）是 Plotly 的「高階 API」，一行就能畫出一張互動圖；但真正做儀表板、多圖排版、加按鈕時，必須改用 `plotly.graph_objects`（`go`）這個「低階 API」。這兩者的關係類似 `sns.lineplot()` 對比 `matplotlib.pyplot`。

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = 'plotly_white'  # 乾淨白底

df = pd.read_csv(
    '../datasets/ecommerce/orders_enriched.csv',
    parse_dates=['order_date'],
)
print('資料形狀:', df.shape)

In [ ]:
# 1) 折線圖：月度營收
df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month', as_index=False)['amount'].sum()

fig = px.line(monthly, x='month', y='amount', markers=True,
              title='Monthly Revenue Trend')
fig.update_layout(height=400)
fig.show()

In [ ]:
# 2) 長條圖：地區營收
region_rev = df.groupby('region', as_index=False)['amount'].sum().sort_values('amount', ascending=False)
fig = px.bar(region_rev, x='region', y='amount', text='amount',
             color='region', title='Revenue by Region')
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# 3) 散佈圖：單價 vs 金額
fig = px.scatter(df, x='unit_price', y='amount',
                 color='category', hover_data=['product_name', 'customer_name'],
                 title='Unit Price vs Order Amount')
fig.update_layout(height=450)
fig.show()

In [ ]:
# 4) 箱型圖：品類分布
fig = px.box(df, x='category', y='amount', color='category',
             title='Order Amount by Category')
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# 5) 直方圖：金額分布
fig = px.histogram(df, x='amount', nbins=30, title='Amount Distribution')
fig.update_layout(height=400)
fig.show()

In [ ]:
# 6) 圓餅圖：VIP 等級佔比
vip_rev = df.groupby('vip_level', as_index=False)['amount'].sum()
fig = px.pie(vip_rev, names='vip_level', values='amount',
             title='VIP Level Share', hole=0.4)
fig.update_layout(height=400)
fig.show()

---
## Part B — Capstone：從 raw 到 dashboard 完整走一遍（50 min）

**情境**：你是公司新進 DA，主管丟給你一份髒的訂單 CSV，下週一要交一份互動式儀表板。  
**目標**：完整走一次 S2（清理）→ S3（合併）→ S4（時序/聚合）→ S6（視覺化）。

> 💡 **教學提示（解答版額外補充）**：真實世界裡，DA 的工作 70% 都在「清理 + 合併」，真正做圖往往只佔 20%。所以請把 Step 1–3 當作重點，Step 4 只是「最後 1 公里」。

### Step 1 — 載入與清理（S2）

把清理邏輯包成函式 `load_and_clean_orders()`，之後延伸挑戰 3 會直接重用這個函式。

In [ ]:
def load_and_clean_orders(path):
    """封裝 S2 的清理邏輯，方便未來重用。

    步驟：欄名標準化 → 金額轉 float → 日期轉 datetime → 缺值處理 → 去重。
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    df['amount'] = (
        df['amount'].astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    df = df.dropna(subset=['order_date'])
    df['qty'] = df['qty'].fillna(df['qty'].median())
    df = df.drop_duplicates()
    return df

orders = load_and_clean_orders('../datasets/ecommerce/orders_raw.csv')
print(f'清理完成：{orders.shape[0]} 筆訂單')
orders.head(3)

### Step 2 — 合併主檔（S3）

In [ ]:
customers = pd.read_csv('../datasets/ecommerce/customers.csv')
products  = pd.read_csv('../datasets/ecommerce/products.csv')

enriched = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print(f'合併後形狀: {enriched.shape}')
print(f'欄位數: {len(enriched.columns)}')

### Step 3 — 計算關鍵指標（S3 + S4）

In [ ]:
enriched['month'] = enriched['order_date'].dt.to_period('M').astype(str)

kpis = {
    '總營收':     enriched['amount'].sum(),
    '總訂單數':   len(enriched),
    '活躍顧客數': enriched['customer_id'].nunique(),
    '客單價':     enriched['amount'].sum() / len(enriched),
}
for k, v in kpis.items():
    print(f'{k}: {v:>12,.0f}')

### Step 4 — 互動式儀表板（4 張圖，2x2 subplot）

> 💡 **教學提示：`make_subplots` 的 `specs` 是什麼？**  
> Plotly 將「圖的座標系統」分成兩大類：  
> - `'xy'`：有 X/Y 軸的圖（折線、長條、散佈、箱型、直方）。  
> - `'domain'`：沒有 X/Y 軸，而是佔一塊區域（圓餅、甜甜圈、sunburst、treemap）。  
> 
> 所以當你想在同一張 figure 放入「3 張 xy 圖 + 1 張圓餅」時，**必須用 `specs` 告訴 Plotly 每一格的類型**，不然它預設每一格都是 `xy`，圓餅圖會畫不出來。  
> 
> **為什麼不直接用 Plotly Express？**  
> `px` 非常方便，但它**不支援在同一張 figure 中混合 xy 與 domain 類型**。一旦你的儀表板要做「折線 + 圓餅 + 長條」這種異質排版，就必須退回 `go + make_subplots` 這條路。

> ### 💡 知識補給站 — 儀表板設計心法：KPI → Trend → Detail
> 
> 好的儀表板有**層次結構**，不是把所有圖塞在一起：
> 
> | 層次 | 放什麼 | 目的 |
> |---|---|---|
> | **最上面** | KPI 卡片（3~5 個數字） | 一眼抓住重點（總營收、訂單數、客單價） |
> | **中間** | 趨勢圖（折線） | 讓老闆知道「是在變好還是變差」 |
> | **下面** | 明細 / 排名（長條、表格） | 哪個地區、品類、商品表現最好 |
> 
> 這叫 **"KPI → Trend → Detail" 三層架構**，也是 Tableau / Power BI 的預設佈局邏輯。
> 
> 我們的 Step 3 算了 KPI、Step 4 就是趨勢 + 明細 — 你已經在做對的事了。
> 
> **注意**：Dashboards ≠ Reports — 儀表板是**持續監控**用的，報告是**解釋過去決策**用的。
> 
> *延伸關鍵字：dashboard design, KPI hierarchy, Tableau, Power BI, information architecture*

In [ ]:
# 先算四份資料
monthly     = enriched.groupby('month', as_index=False)['amount'].sum()
top_prod    = (enriched.groupby('product_name', as_index=False)['amount']
               .sum().sort_values('amount', ascending=False).head(10))
region_rev  = enriched.groupby('region', as_index=False)['amount'].sum()
cat_rev     = enriched.groupby('category', as_index=False)['amount'].sum()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Revenue Trend',
                    'Top 10 Products',
                    'Revenue by Region',
                    'Category Share'),
    specs=[[{'type': 'xy'},     {'type': 'xy'}],
           [{'type': 'xy'},     {'type': 'domain'}]],
)

fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['amount'],
                         mode='lines+markers', name='Monthly'), row=1, col=1)
fig.add_trace(go.Bar(x=top_prod['product_name'], y=top_prod['amount'],
                     name='Top Products'), row=1, col=2)
fig.add_trace(go.Bar(x=region_rev['region'], y=region_rev['amount'],
                     name='Region'), row=2, col=1)
fig.add_trace(go.Pie(labels=cat_rev['category'], values=cat_rev['amount'],
                     name='Category', hole=0.4), row=2, col=2)

fig.update_layout(
    title_text='<b>E-Commerce Sales Dashboard — 2025【解答版】</b>',
    height=750, showlegend=False,
)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.show()

### Step 5 — 存檔：把儀表板變成可以寄出去的 HTML

> 💡 **教學提示：HTML 成品怎麼交付 / 部署？**  
> `fig.write_html()` 產生的是一份**完全獨立的 HTML 檔案**（預設會把 plotly.js 以 CDN 方式載入），你可以：  
> 1. **直接寄 email** — 老闆點兩下就能在瀏覽器中互動。  
> 2. **丟 S3 / GCS / Azure Blob** — 配合 static website hosting，產生一個公開網址。  
> 3. **塞到公司內部 Confluence / Notion** — 多數知識庫都支援嵌入 HTML iframe。  
> 4. **升級為 Web App** — 當互動需求變複雜（需要後端計算、使用者登入），再轉用 **Streamlit**、**Dash** 或 **Gradio**。

In [ ]:
fig.write_html('../datasets/ecommerce/dashboard_solution.html')
print('✅ 儀表板已存成 dashboard_solution.html，可直接寄給老闆。')

---
## 延伸挑戰 1 — 時間篩選器（`updatemenus`）

讓使用者在月度趨勢圖上切換「全年 / Q1 / Q2 / Q3 / Q4」視角。

> 💡 **觀念**：`updatemenus` 是 Plotly 的「按鈕 / 下拉選單」機制。每個按鈕本質上是一組 `args`，告訴 Plotly：「按下去之後，請把這些圖表屬性更新成這些值」。  
> 常見寫法有兩種：  
> - **切換 trace 顯示** — 用 `{'visible': [True, False, ...]}` 決定哪些 trace 要顯示。  
> - **切換資料本身** — 用 `{'x': [...], 'y': [...]}` 直接換掉 trace 的 x/y 數值（本節使用這種）。

In [ ]:
# 準備每一個季度的資料
enriched['quarter'] = enriched['order_date'].dt.quarter

monthly_all = enriched.groupby('month', as_index=False)['amount'].sum()

def monthly_by_quarter(q):
    sub = enriched[enriched['quarter'] == q]
    return sub.groupby('month', as_index=False)['amount'].sum()

q_frames = {q: monthly_by_quarter(q) for q in [1, 2, 3, 4]}

fig_q = go.Figure()
fig_q.add_trace(go.Scatter(
    x=monthly_all['month'], y=monthly_all['amount'],
    mode='lines+markers', name='Monthly Revenue',
    line=dict(width=3),
))

def _btn(label, df_view):
    return dict(
        label=label,
        method='update',
        args=[{'x': [df_view['month'].tolist()],
               'y': [df_view['amount'].tolist()]},
              {'title': f'Monthly Revenue Trend — {label}'}],
    )

buttons = [
    _btn('全年', monthly_all),
    _btn('Q1',   q_frames[1]),
    _btn('Q2',   q_frames[2]),
    _btn('Q3',   q_frames[3]),
    _btn('Q4',   q_frames[4]),
]

fig_q.update_layout(
    title='Monthly Revenue Trend — 全年',
    height=450,
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.5, y=1.18, xanchor='center',
        showactive=True,
        buttons=buttons,
    )],
)
fig_q.show()

---
## 延伸挑戰 2 — RFM 3D 散佈圖（`px.scatter_3d`）

> 💡 **RFM 是什麼？** 顧客分群中最經典的 3 個指標：  
> - **Recency（R）**：上次購買距今幾天，越小越活躍。  
> - **Frequency（F）**：購買次數，越大越忠誠。  
> - **Monetary（M）**：總消費金額，越大越有貢獻度。  
> 
> 把每一位顧客畫成 3D 空間中的一個點，就能一眼看出「誰是 VIP、誰是流失風險」。

In [ ]:
# 用 enriched 算 RFM
snapshot_date = enriched['order_date'].max() + pd.Timedelta(days=1)

rfm = (
    enriched
    .groupby(['customer_id', 'customer_name', 'vip_level'], as_index=False)
    .agg(
        recency=('order_date', lambda s: (snapshot_date - s.max()).days),
        frequency=('order_date', 'count'),
        monetary=('amount', 'sum'),
    )
)
print(f'顧客數: {len(rfm)}')
rfm.head()

In [ ]:
fig_rfm = px.scatter_3d(
    rfm,
    x='recency', y='frequency', z='monetary',
    color='vip_level',
    hover_data=['customer_name'],
    title='RFM 3D Segmentation by VIP Level',
    labels={'recency': 'Recency (days)',
            'frequency': 'Frequency',
            'monetary': 'Monetary ($)'},
)
fig_rfm.update_traces(marker=dict(size=5, opacity=0.75))
fig_rfm.update_layout(height=600)
fig_rfm.show()

---
## 延伸挑戰 3 — 可重用 Pipeline：`generate_dashboard()`

把整條 pipeline 打包成一個函式：

```python
generate_dashboard(
    orders_path='../datasets/ecommerce/orders_raw.csv',
    customers_path='../datasets/ecommerce/customers.csv',
    products_path='../datasets/ecommerce/products.csv',
    output_html='../datasets/ecommerce/dashboard_solution.html',
)
```

之後每個月只要換 CSV 路徑，就能一鍵產出新的 HTML 儀表板 — 這就是「資料產品化」的雛形。

In [ ]:
def generate_dashboard(orders_path, customers_path, products_path, output_html):
    """端到端 pipeline：載入 → 清理 → 合併 → 繪製 4 圖儀表板 → 輸出 HTML。

    Parameters
    ----------
    orders_path : str
        原始訂單 CSV 路徑。
    customers_path : str
        顧客主檔 CSV 路徑。
    products_path : str
        商品主檔 CSV 路徑。
    output_html : str
        輸出 HTML 檔案的路徑。

    Returns
    -------
    plotly.graph_objects.Figure
        產生的儀表板 Figure，方便呼叫端再做二次修改。
    """
    # 1. 載入 + 清理
    orders_df = load_and_clean_orders(orders_path)
    customers_df = pd.read_csv(customers_path)
    products_df = pd.read_csv(products_path)

    # 2. 合併
    data = (
        orders_df
        .merge(customers_df, on='customer_id', how='left')
        .merge(products_df,  on='product_id',  how='left')
    )
    data['month'] = data['order_date'].dt.to_period('M').astype(str)

    # 3. 聚合四份資料
    monthly_df = data.groupby('month', as_index=False)['amount'].sum()
    top_prod_df = (data.groupby('product_name', as_index=False)['amount']
                   .sum().sort_values('amount', ascending=False).head(10))
    region_df = data.groupby('region', as_index=False)['amount'].sum()
    cat_df = data.groupby('category', as_index=False)['amount'].sum()

    # 4. 組合 2x2 subplot
    dash = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Monthly Revenue Trend',
                        'Top 10 Products',
                        'Revenue by Region',
                        'Category Share'),
        specs=[[{'type': 'xy'}, {'type': 'xy'}],
               [{'type': 'xy'}, {'type': 'domain'}]],
    )
    dash.add_trace(go.Scatter(x=monthly_df['month'], y=monthly_df['amount'],
                              mode='lines+markers', name='Monthly'),
                   row=1, col=1)
    dash.add_trace(go.Bar(x=top_prod_df['product_name'],
                          y=top_prod_df['amount'], name='Top'),
                   row=1, col=2)
    dash.add_trace(go.Bar(x=region_df['region'], y=region_df['amount'],
                          name='Region'), row=2, col=1)
    dash.add_trace(go.Pie(labels=cat_df['category'], values=cat_df['amount'],
                          hole=0.4, name='Category'), row=2, col=2)
    dash.update_layout(
        title_text='<b>E-Commerce Sales Dashboard【auto-generated】</b>',
        height=750, showlegend=False,
    )
    dash.update_xaxes(tickangle=45, row=1, col=2)

    # 5. 輸出 HTML
    dash.write_html(output_html)
    print(f'✅ Dashboard 已輸出：{output_html}')
    return dash


# 範例呼叫（與前面 Step 4/5 效果相同，但只用一行）
_ = generate_dashboard(
    orders_path='../datasets/ecommerce/orders_raw.csv',
    customers_path='../datasets/ecommerce/customers.csv',
    products_path='../datasets/ecommerce/products.csv',
    output_html='../datasets/ecommerce/dashboard_solution.html',
)

---
## 小結與速查表

| 想畫什麼 | Plotly Express / graph_objects |
|---|---|
| 折線 | `px.line(df, x, y, markers=True)` |
| 長條 | `px.bar(df, x, y, color, text)` |
| 散佈 | `px.scatter(df, x, y, color, hover_data)` |
| 3D 散佈 | `px.scatter_3d(df, x, y, z, color)` |
| 箱型 | `px.box(df, x, y, color)` |
| 直方 | `px.histogram(df, x, nbins)` |
| 圓餅 | `px.pie(df, names, values, hole=0.4)` |
| 多圖 | `make_subplots(specs=...) + add_trace` |
| 按鈕切換 | `fig.update_layout(updatemenus=[...])` |
| 存檔 | `fig.write_html(path)` |

### 解答版三個延伸挑戰的學習重點

1. **`updatemenus`**：了解 Plotly 的按鈕更新機制，能把一張「靜態」趨勢圖升級為「可切換視角」的互動圖。
2. **`px.scatter_3d` + RFM**：把顧客分群與 3D 視覺化結合，展示 Plotly 在多維資料上的優勢。
3. **`generate_dashboard()`**：從「在 notebook 中寫分析」進化到「把分析包成函式」，是從 DA 走向 Data Product 的關鍵一步。

**記住一件事：DA/DE 的價值不在技術，在於你能回答老闆『所以呢？』這個問題。**

> ### 💡 知識補給站 — 從分析到行動：ML 與 A/B Test 的下一步
> 
> 本課程你學會了「**描述過去**」（Descriptive Analytics）— 過去賣了多少、哪個品類最好、哪個月最旺。
> 
> 但真正有商業價值的是**下一步**：
> 
> - **預測未來（Predictive）**：用今天算的特徵（月營收、客單價、RFM）餵給 **scikit-learn**，預測「這個客戶下個月會不會回購？」→ 這就是 **Feature Engineering**，你在 S3 做的 groupby + agg 就是在算特徵
> - **驗證假設（Prescriptive）**：你發現某地區營收特別高，但這是因為行銷活動還是自然需求？→ 設計 **A/B Test**，用統計檢定（t-test / chi-square）驗證差異是否顯著
> 
> 記住：**DA 的終極價值不是做圖，是回答「所以我們該怎麼做？」**
> 
> *延伸關鍵字：feature engineering, scikit-learn, A/B testing, t-test, descriptive → predictive → prescriptive analytics*